# Notebook 03 — Recuperador KNN / TF-IDF

Implementa o recuperador denso usando TF-IDF + cosine similarity.
Gera o arquivo `runs/knn.trec` no formato TREC.

In [ ]:
import sys
sys.path.insert(0, '..')
from src.retrievers import build_tfidf_index, search_knn
from src.utils import load_corpus, load_queries, write_trec_run
from pathlib import Path

CORPUS_PATH  = '../data/corpus.jsonl'
QUERIES_PATH = '../eval/queries.tsv'
OUTPUT_PATH  = 'runs/knn.trec'
K = 100

corpus  = load_corpus(CORPUS_PATH)
queries = load_queries(QUERIES_PATH)
print(f'Corpus: {len(corpus)} documentos | Queries: {len(queries)}')

In [ ]:
print('Construindo índice TF-IDF...')
vectorizer, matrix = build_tfidf_index(corpus)
print(f'Vocabulário: {len(vectorizer.vocabulary_)} termos | Matriz: {matrix.shape}')

In [ ]:
Path('runs').mkdir(exist_ok=True)
# Limpar arquivo de saída
open(OUTPUT_PATH, 'w').close()

for qid, qtext in queries.items():
    results = search_knn(qtext, vectorizer, matrix, corpus, k=K)
    write_trec_run(results, qid, 'knn_tfidf', OUTPUT_PATH)
    print(f'{qid}: top-1 = {results[0][0]} ({results[0][1]:.4f})')

print(f'\nSalvo em {OUTPUT_PATH}')

## Busca de exemplo

Dada uma consulta em texto livre, retorna o top-10.

In [ ]:
DEMO_QUERY = 'public procurement document classification machine learning'
results = search_knn(DEMO_QUERY, vectorizer, matrix, corpus, k=10)

# Mapa id -> doc para exibição
id2doc = {d['arxiv_id']: d for d in corpus}
print(f'Query: "{DEMO_QUERY}"\n')
for rank, (doc_id, score) in enumerate(results, 1):
    doc = id2doc.get(doc_id, {})
    print(f'{rank:2}. [{score:.4f}] {doc_id}')
    print(f'    {doc.get("title", "sem título")[:100]}')